# 🎨 WAI-Illustrious v17 — Google Colab · 1.0.3
### Forge Neo + tagcomplete + Prompt VN + 349.714 tag / 45.513 bản dịch

Notebook này dùng **cùng bộ cài đã kiểm tra** với notebook một ô, nhưng chia từng
stage để dễ theo dõi lỗi. Không ComfyUI, không mount Drive, không tự đổi model.

**Chạy mới từ đầu:**
1. Chọn **Runtime → Change runtime type → GPU T4** (hoặc GPU NVIDIA khác).
2. Chọn model ở Ô 1; mặc định **FP16 gốc**, checkpoint tại `/content/models`.
3. **Runtime → Run all**, hoặc chạy lần lượt Ô 1 → Ô 8. Forge chạy nền, Ô 8 có thể
   kết thúc sau launch/verify; không bị chặn ở một ô chạy mãi.
4. Mở link public và dùng tab **Prompt VN**. Ô 9/10/11 mặc định chỉ in hướng dẫn,
   không tự sinh ảnh hay mở thêm server.

Nếu muốn môi trường hoàn toàn mới khi đang dùng phiên cũ, **tải ảnh về trước** rồi
chọn **Runtime → Disconnect and delete runtime**, kết nối lại và bật GPU. Notebook
không tự xóa thư mục cũ. Chạy lại trong cùng phiên sẽ tiếp tục/cập nhật an toàn.

| Model | Ghi chú |
|---|---|
| `fp16` ⭐ | WAI v17 gốc pruned, 6,46 GiB; VAE tích hợp; SHA256 đã biết; khuyên dùng |
| `fp8` / `int8` | Bản lượng tử hóa cộng đồng, thử nghiệm; CFG 6.5 / 25 bước |
| `hswq` | Turbo merge cộng đồng, thử nghiệm; CFG 1 / 10 bước, không dùng CFG FP16 |

Cần khoảng **25 GB trống**, **10–25 phút** cho lần cài đầu tùy mạng/runtime. T4 không
có tensor core FP8; model nhỏ hơn không bảo đảm nhanh hơn. Chưa benchmark GPU thật.
Dữ liệu `/content` mất khi kết thúc runtime; không có thủ thuật giữ runtime luôn hoạt động.
Mặc định phân loại `general`; negative prompt không bảo đảm lọc mọi nội dung.
Tuân thủ giấy phép model, pháp luật và điều khoản Google Colab.


## ⚙️ Ô 1 — Cấu hình

FP16 được khuyên dùng. Nếu cần token tải Civitai, thêm `CIVITAI_TOKEN` vào **Colab
Secrets**, bật Notebook access và `USE_CIVITAI_SECRET`; không dán token vào code.
Repo public không cần GitHub token. Thư mục mặc định đều trong `/content`.


In [ ]:
#@title ⚙️ Model và thư mục (cài mới) { display-mode: "form" }
MODEL = "fp16" #@param ["fp16", "fp8", "int8", "hswq"]
MODEL_DIR = "/content/models" #@param {type:"string"}
TUNNEL = "cloudflared" #@param ["cloudflared", "share", "none"]
PORT = 7860 #@param {type:"integer"}
CUDA_INDEX = "auto" #@param ["auto", "cu130", "cu128", "cu126"]
RESTART = False #@param {type:"boolean"}
FALLBACK_FP16 = False #@param {type:"boolean"}
USE_CIVITAI_SECRET = False #@param {type:"boolean"}
USE_GITHUB_SECRET = False #@param {type:"boolean"}

import os
os.environ.update({
    "REPO_DIR": "/content/ai-anime3",
    "REPO_URL": "https://github.com/manhlee1196-boop/ai-anime3",
    "REPO_BRANCH": "arena/88f01d31-ai-anime3",
    "FORGE_DIR": "/content/sd-webui-forge-neo",
    "FORGE_REPO": "https://github.com/Haoming02/sd-webui-forge-classic",
    "FORGE_BRANCH": "neo",
    "MODEL_ID": MODEL, "MODEL_DIR": MODEL_DIR, "TUNNEL": TUNNEL,
    "PORT": str(PORT), "CUDA_INDEX": CUDA_INDEX,
    "VI_PROMPT_ROOT": "/content/ai-anime3",
})
# Không giữ MODEL_FILE/SHA/URL từ ô cấu hình FP16 của notebook cũ khi chọn FP8/HSWQ.
# Muốn mirror/checkpoint tùy chỉnh, dùng cờ bộ cài trong colab/INSTALL.md.
for key in ("MODEL_FILE", "MODEL_SHA256", "MODEL_URL", "MODEL_MIN_BYTES"):
    os.environ.pop(key, None)
if USE_CIVITAI_SECRET or USE_GITHUB_SECRET:
    from google.colab import userdata
    for enabled, name in ((USE_CIVITAI_SECRET, "CIVITAI_TOKEN"), (USE_GITHUB_SECRET, "GITHUB_TOKEN")):
        if enabled:
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                raise RuntimeError(f"Chưa đọc được Secret {name}; thêm secret và bật Notebook access.") from None
print(f"✔ {MODEL} · model {MODEL_DIR} · Forge {os.environ['FORGE_DIR']} · {TUNNEL}:{PORT}")

## Ô 2 — Lấy bộ cài mới và repo đúng nhánh

Bootstrap thử raw GitHub rồi API, kiểm **phiên bản ≥1.0.3** và cú pháp trước khi
chạy. Không chạy cache 1.0.2 cũ. Bộ cài clone đầy đủ kho prompt + thư viện mới + JS
thay Việt → Anh; repo sẵn có được fast-forward khi sạch/cùng nhánh, không reset hard.
Không đổi sang main/master hoặc xóa repo có thay đổi của bạn.


In [ ]:
import os, subprocess
import re
import subprocess
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

MIN_INSTALLER_VERSION = "1.0.3"


def checked_installer_version(data):
    if not data.startswith(b"#!/usr/bin/env bash") or len(data) < 5000:
        raise ValueError("Không phải script bộ cài đầy đủ")
    match = re.search(rb'^INSTALLER_VERSION=[\'"](\d+\.\d+\.\d+)[\'"]$', data, re.M)
    if not match:
        raise ValueError("Không đọc được phiên bản bộ cài")
    version = match[1].decode("ascii")
    if tuple(map(int, version.split("."))) < tuple(map(int, MIN_INSTALLER_VERSION.split("."))):
        raise ValueError(f"Bộ cài quá cũ; cần từ {MIN_INSTALLER_VERSION}")
    return version


def download_installer(dest, branch, env):
    owner_repo = "manhlee1196-boop/ai-anime3"
    raw = f"https://raw.githubusercontent.com/{owner_repo}/{branch}/colab/install.sh"
    api = (f"https://api.github.com/repos/{owner_repo}/contents/colab/install.sh?ref="
           + urllib.parse.quote(branch, safe=""))
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".sh.tmp")
    for url in (raw, api):
        headers = {"Accept": "application/vnd.github.raw", "User-Agent": "wai-colab-installer/1.0.3"}
        if env.get("GITHUB_TOKEN"):
            headers["Authorization"] = f"Bearer {env['GITHUB_TOKEN']}"
        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=30) as response:
                data = response.read()
            version = checked_installer_version(data)
            tmp.write_bytes(data)
            subprocess.run(["bash", "-n", str(tmp)], check=True, capture_output=True)
            tmp.replace(dest)
            print(f"✔ Đã lấy bộ cài {version} từ nhánh {branch}.", flush=True)
            return version
        except (OSError, ValueError, subprocess.CalledProcessError, urllib.error.URLError):
            continue
        finally:
            if tmp.exists():
                tmp.unlink()  # chỉ file tạm do ô này tạo; không đụng model/repo/ảnh
    # Mất mạng chỉ dùng cache nếu phiên bản và cú pháp đều hợp lệ. Không chạy 1.0.2.
    try:
        version = checked_installer_version(dest.read_bytes())
        subprocess.run(["bash", "-n", str(dest)], check=True, capture_output=True)
    except (OSError, ValueError, subprocess.CalledProcessError):
        raise RuntimeError(
            f"Không lấy được bộ cài >= {MIN_INSTALLER_VERSION}. Kiểm tra mạng hoặc mở lại notebook mới; "
            "chưa chạy cài đặt và không xóa dữ liệu cũ."
        ) from None
    print(f"⚠ Không lấy được bản mới; dùng cache đã kiểm tra {version} trong /content.", flush=True)
    return version

INSTALLER = Path("/content/install.sh")
download_installer(INSTALLER, os.environ["REPO_BRANCH"], dict(os.environ))


def run_install(stages, launch=False, restart=False, smoke=False):
    args = ["bash", str(INSTALLER), "--only", stages, "--model", MODEL,
            "--model-dir", MODEL_DIR, "--branch", os.environ["REPO_BRANCH"],
            "--port", str(PORT), "--tunnel", TUNNEL, "--cuda", CUDA_INDEX]
    if not launch:
        args.append("--no-launch")
    if restart:
        args.append("--restart")
    if smoke:
        args.append("--smoke-test")
    if FALLBACK_FP16:
        args.append("--fallback-fp16")
    subprocess.run(args, env=dict(os.environ), check=True)


run_install("repo")
# Stage sau ưu tiên setup_forge.sh trong repo. Từ chối trộn checkout/bộ cài cũ.
checked_installer_version((Path(os.environ["REPO_DIR"]) / "colab/install.sh").read_bytes())
print("✔ Repo/bộ cài đủ phiên bản; tiếp tục Ô 3 → Ô 8.")

## Ô 3 — Kiểm tra GPU, driver và ổ đĩa

Kiểm NVIDIA GPU, đường dẫn, dung lượng và mạng. Chưa tải checkpoint hay cài Torch.


In [ ]:
run_install("preflight")

## Ô 4 — Cài Forge Neo

Dùng Python 3.13 riêng qua uv; giữ thư mục và cấu hình sẵn có.


In [ ]:
run_install("forge")

## Ô 5 — Cài PyTorch phù hợp GPU

Bộ cài chọn cu130/cu128/cu126 theo driver và kiểm kernel CUDA, không chỉ cuda.is_available().


In [ ]:
run_install("torch")

## Ô 6 — Tải và xác minh checkpoint WAI v17

Tải tiếp .part; xác minh SHA256 + header safetensors. Chỉ đúng biến thể/file v17, không tự đổi model khi nguồn lỗi.


In [ ]:
run_install("model")

## Ô 7 — Cắm tagcomplete + Prompt VN mới

Chép CSV/JS mới, cấu hình dịch 2 cột, CLIP stop 2, styles và upscaler anime. Thư viện editing/workflow phải có đủ trước khi thay extension.


In [ ]:
run_install("extras")

## Ô 8 — Khởi chạy nền + kiểm tra checkpoint/API

Chờ bộ cài in link `trycloudflare.com` hoặc `gradio.live`, mở link và chọn **Prompt VN**.
Ô này kết thúc khi API/verify đạt; Forge/tunnel tiếp tục chạy nền trong runtime.
Không tự sinh ảnh để chứng minh trọng số đã nạp; bước đó là tùy chọn Ô 9.
Nếu dùng phiên cũ, bật `RESTART=True` ở Ô 1 để nạp extension/JS mới rồi reload cứng UI.
### Quy trình mới trong Prompt VN (1.0.3)

1. Nhập `1girl, (áo dài:1.2), tóc dài`, chọn kết quả hoặc bấm **Thay mục Việt → tag Anh**.
   Kết quả thay tại chỗ: `1girl, (ao_dai:1.2), long_hair`; giữ tag khác/trọng số,
   không giữ cả chữ Việt + Anh hoặc tự xóa toàn bộ prompt.
2. **Dựng prompt**, bắt đầu **Nháp** hoặc **Chuẩn**, mặc định **832×1216, Hires tắt**.
3. Chọn một ảnh trong Gallery → **Dùng seed của ảnh đã chọn** (không chọn ảnh lưới).
4. Chuyển **Hoàn thiện**: Hires 1.5×, denoise 0.35, tự đặt **1 ảnh/lượt** và giữ sampler.
   Preset áp lại steps/CFG; cùng seed không bảo đảm ảnh giữ nguyên khi thay tham số.

| Mức | FP16 | FP8/INT8 | HSWQ | Hires |
|---|---|---|---|---|
| Nháp | 18 bước / CFG 6 | 18 / 6.5 | 10 / 1 | Tắt |
| Chuẩn | 25 / 6 | 25 / 6.5 | 10 / 1 | Tắt |
| Hoàn thiện | 25 / 6 | 25 / 6.5 | 10 / 1 | 1.5×, denoise .35; 20 bước (HSWQ 10) |

Nhiều ảnh chạy tuần tự (`batch_size=1`), không tăng batch để tiết kiệm VRAM.
Đây là preset ứng dụng, chưa phải benchmark tốc độ/chất lượng trên GPU Colab.
**Sinh ảnh** trong Prompt VN dùng các tham số trên; **Đưa vào txt2img** chỉ chuyển
prompt, không chuyển sampling. Điền txt2img mặc định ghép/thay mục tương ứng;
**Thay toàn bộ** phải bật rõ ràng và có **Hoàn tác** trong phiên trình duyệt.


In [ ]:
run_install("launch,verify", launch=True, restart=RESTART)

## 🔧 Ô 9 (tùy chọn) — Chẩn đoán và sinh ảnh kiểm tra

Chỉ bật `RUN_DIAGNOSTICS` khi cần xem checkpoint/preset/API. `SMOKE_TEST` dùng GPU
để nạp trọng số và sinh **1 ảnh 256×256 / 1 step**, không phải benchmark/chất lượng.
Mặc định cả hai tắt nên **Run all không tự sinh ảnh**.


In [ ]:
#@title 🔧 Kiểm tra thêm (tùy chọn) { display-mode: "form" }
RUN_DIAGNOSTICS = False #@param {type:"boolean"}
SMOKE_TEST = False #@param {type:"boolean"}
if RUN_DIAGNOSTICS:
    subprocess.run(["bash", str(INSTALLER), "--diagnostics"], check=True)
if SMOKE_TEST:
    run_install("verify", launch=True, smoke=True)
if not RUN_DIAGNOSTICS and not SMOKE_TEST:
    print("Bỏ qua chẩn đoán/sinh ảnh thử. Launch + verify đã chạy ở Ô 8.")

## 🖼️ Ô 10 (tùy chọn) — Sinh ảnh bằng API theo quy trình mới

Bật `RUN_API_SAMPLE` khi muốn dùng API thay cho tab Prompt VN. Mặc định **Nháp,
832×1216, Hires tắt**, steps/CFG theo model đã chọn. Nhiều ảnh chạy tuần tự.
Sau lượt Nháp/Chuẩn, chọn seed của ảnh thật (không phải grid), nhập `SEED` và chọn
Hoàn thiện để sinh 1 ảnh Hires. Mã dùng bộ giải mã chung cho info JSON-string và
PNG/PIL, không dùng ví dụ API cũ bật Hires ngay hoặc cố định CFG FP16 cho HSWQ.


In [ ]:
#@title 🖼️ Ví dụ API — chỉ sinh khi bật RUN_API_SAMPLE { display-mode: "form" }
RUN_API_SAMPLE = False #@param {type:"boolean"}
MODE = "Nháp" #@param ["Nháp", "Chuẩn", "Hoàn thiện"]
SELECTED_TAGS = "1girl, (áo dài:1.2), tóc dài, cherry_blossoms, looking_at_viewer" #@param {type:"string"}
SEED = -1 #@param {type:"integer"}
IMAGE_COUNT = 1 #@param {type:"integer"}


def generate_notebook_sample():
    import os
    import sys
    from datetime import datetime
    from pathlib import Path
    import requests

    repo = Path(os.environ["REPO_DIR"])
    sys.path.insert(0, str(repo / "colab"))
    from prompt_lib import PromptBuilder, TagIndex, WAI_DEFAULTS
    from prompt_lib.api import decode_generation_response
    from prompt_lib.editing import convert_tags
    from prompt_lib.workflow import (WORKFLOW_PRESETS, workflow_settings, validate_generation,
                                     generation_metadata, generation_error, seed_at)

    label = next(label for label, mode in WORKFLOW_PRESETS.items()
                 if mode == {"Nháp": "draft", "Chuẩn": "standard", "Hoàn thiện": "final"}[MODE])
    steps, cfg, sampler, hires, scale, hr_steps, denoise = workflow_settings(label)
    if hires and SEED == -1:
        raise ValueError("Hoàn thiện: nhập seed của ảnh đã chọn ở lần Nháp/Chuẩn trước; không dùng seed ảnh lưới.")
    index = TagIndex(repo)
    builder = PromptBuilder(index)
    converted, changes, unknown = convert_tags(SELECTED_TAGS, index)
    print("Tag sau thay Việt → Anh:", converted)
    if unknown:
        print("Mục chưa rõ được giữ nguyên:", ", ".join(unknown))
    out = builder.assemble(converted, rating_id="general", quality_preset="Ngắn gọn")
    negative = builder.negative("general", "WAI khuyến nghị (ngắn)")
    payload = validate_generation(builder.payload(out["prompt"], negative, {
        "steps": steps, "cfg_scale": cfg, "sampler_name": sampler, "seed": SEED,
        "width": 832, "height": 1216, "enable_hr": hires, "hr_scale": scale,
        "hr_second_pass_steps": hr_steps, "denoising_strength": denoise,
        "batch_size": 1, "n_iter": 1 if hires else IMAGE_COUNT,
        "override_settings": {"clip_stop_at_last_layers": WAI_DEFAULTS["clip_skip"]},
        "save_images": True, "send_images": True,
    }))
    url = f'http://127.0.0.1:{os.environ.get("PORT", "7860")}'
    print(f"▶ {MODE}: {steps} bước / CFG {cfg} / {sampler} / Hires {hires}; {payload['n_iter']} ảnh tuần tự.")
    try:
        if hires:
            response = requests.get(url + "/sdapi/v1/upscalers", timeout=15)
            response.raise_for_status()
            builder.fix_upscaler(payload, [item["name"] for item in response.json() if item.get("name")])
        response = requests.post(url + "/sdapi/v1/txt2img", json=payload, timeout=1800)
        response.raise_for_status()
        images, info = decode_generation_response(response.json())  # info JSON-string + PNG → PIL
    except Exception as exc:
        raise RuntimeError(generation_error(exc)) from None
    metadata = generation_metadata(info, payload, len(images))
    output = repo / "colab" / "outputs"
    output.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    paths = []
    for i, image in enumerate(images):
        image_seed = seed_at(metadata, i)
        suffix = f"seed{image_seed}" if image_seed is not None else "grid-or-unknown-seed"
        path = output / f"wai17_{stamp}_{i}_{suffix}.png"
        image.save(path, format="PNG")
        paths.append(str(path))
        print("✔ Đã lưu", path)
    print("Seed từng ảnh (trừ grid):", metadata["seeds"], "· grid offset:", metadata["grid_offset"])
    print("Ảnh /content mất khi kết thúc runtime; tải về trước khi ngắt.")
    return {"metadata": metadata, "paths": paths, "converted_tags": converted}


if RUN_API_SAMPLE:
    API_LAST_RESULT = generate_notebook_sample()
else:
    print("Bỏ qua sinh ảnh API. Dùng tab Prompt VN, hoặc bật RUN_API_SAMPLE khi Forge đã chạy.")


## 🧩 Ô 11 (tùy chọn) — App Prompt VN độc lập

Không cần ô này để dùng tab Prompt VN đã cài trong Forge. Nếu muốn app riêng,
bật `START_PROMPT_APP`; ô kiểm tra PID/cổng trước để tránh chạy server trùng.
App dùng venv Forge, không cài lại Gradio/Torch. Chỉ dùng một nơi để sinh ảnh tại
một thời điểm; queue của app riêng không chặn mọi client GPU khác.


In [ ]:
#@title 🧩 App soạn prompt riêng (không tự mở khi Run all) { display-mode: "form" }
START_PROMPT_APP = False #@param {type:"boolean"}


def start_prompt_app():
    import os, socket, subprocess, time
    from pathlib import Path
    repo = Path(os.environ["REPO_DIR"])
    python = Path(os.environ["FORGE_DIR"]) / "venv/bin/python"
    script = repo / "colab/prompt_builder.py"
    pid_file = Path("/content/prompt_builder.pid")
    log = Path("/content/prompt_builder.log")
    if not python.is_file() or not script.is_file():
        raise RuntimeError("Chưa có Forge/app; chạy Ô 1 → Ô 8 trước.")
    owned = False
    try:
        pid = int(pid_file.read_text().strip())
        parts = Path(f"/proc/{pid}/cmdline").read_bytes().split(b"\0")
        owned = os.fsencode(script) in parts
    except (OSError, ValueError):
        pass
    if not owned:
        with socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", 7861)) == 0:
                raise RuntimeError("Cổng 7861 do dịch vụ khác dùng; không tự đóng dịch vụ đó.")
        env = dict(os.environ)
        for key in ("GITHUB_TOKEN", "CIVITAI_TOKEN"):
            env.pop(key, None)
        with log.open("w") as stream:
            process = subprocess.Popen(
                [str(python), "-u", str(script), "--forge-url", f"http://127.0.0.1:{PORT}",
                 "--host", "0.0.0.0", "--port", "7861", "--share"],
                cwd=repo, env=env, stdout=stream, stderr=subprocess.STDOUT,
                stdin=subprocess.DEVNULL, start_new_session=True,
            )
        pid_file.write_text(str(process.pid))
    else:
        print("✔ App do notebook quản lý đang chạy; không mở server trùng.")
    for _ in range(90):
        text = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
        link = next((line.strip() for line in text.splitlines() if "gradio.live" in line), None)
        if link:
            print("✔ Link app:", link)
            return
        if "Traceback" in text:
            raise RuntimeError("App lỗi; xem /content/prompt_builder.log riêng, không chia sẻ log có dữ liệu riêng.")
        if not owned and process.poll() is not None:
            raise RuntimeError("App đã dừng; xem /content/prompt_builder.log.")
        time.sleep(2)
    raise RuntimeError("Chưa có link sau 180 giây; xem log trước khi thử lại, không mở trùng process.")


if START_PROMPT_APP:
    start_prompt_app()
else:
    print("Bỏ qua app riêng. Dùng tab Prompt VN trong link Forge ở Ô 8.")

---
## 🚑 Xử lý sự cố / chạy lại

| Triệu chứng | Xử lý |
|---|---|
| Không có GPU | Runtime → Change runtime type → GPU T4 rồi chạy lại từ Ô 1 |
| Không lấy được bộ cài ≥1.0.3 | Kiểm mạng/mở notebook mới; không dùng cache cũ 1.0.2 |
| Checkpoint trống / lỗi CUDA | Chạy `!bash /content/install.sh --diagnostics`; gửi phần lỗi đã che secret, không dump config/log riêng |
| Model SHA/header sai | Chạy lại Ô 6 để tải tiếp/thử đúng nguồn; không đổi MODEL_FILE thủ công |
| Tag Việt còn sau chọn | Kiểm tab Prompt VN có nút Thay mục Việt; chạy lại Ô 2 + Ô 7, bật RESTART rồi Ô 8 và reload cứng UI |
| Hết VRAM | Tắt Hires, 832×1216/1024×1024, batch 1; chờ tác vụ GPU khác, không cần tải lại checkpoint |
| Sửa phần mềm trong phiên đang có | `!bash /content/install.sh --only repo,extras,launch,verify --restart` |
| Xem link/trạng thái | `!bash /content/install.sh --status` |
| Chạy mới hoàn toàn | Tải ảnh về, Disconnect and delete runtime, kết nối lại + GPU rồi Run all notebook mới |

Chạy lại stage không xóa model/ảnh/repo. Kho tag chính vẫn là bản Anh có `_`;
file dịch 2 cột, **old 3-column format tắt**. Mọi dữ liệu `/content` là tạm thời.
